In [2]:
import os
import yaml
import shutil
import random
from glob import glob

In [3]:
# Prepare dataset for YOLO training
def prepare_yolo_dataset(data_dir="data", output_dir="yolo_dataset"):
    # Create directory structure
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)
    
    # Collect all image-label pairs from country directories
    all_pairs = []
    country_dirs = [d for d in os.listdir(data_dir) if d.startswith('country_')]
    
    for country_dir in country_dirs:
        images_path = f"{data_dir}/{country_dir}/images"
        labels_path = f"{data_dir}/{country_dir}/labels"
        
        for img_file in glob(f"{images_path}/*.jpg") + glob(f"{images_path}/*.png"):
            base_name = os.path.splitext(os.path.basename(img_file))[0]
            label_file = f"{labels_path}/{base_name}.txt"
            if os.path.exists(label_file):
                all_pairs.append((img_file, label_file))
    
    # Split dataset (70% train, 15% val, 15% test)
    random.seed(42)
    random.shuffle(all_pairs)
    
    n = len(all_pairs)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)
    
    splits = {
        'train': all_pairs[:train_end],
        'val': all_pairs[train_end:val_end],
        'test': all_pairs[val_end:]
    }
    
    # Copy files to respective directories
    for split_name, pairs in splits.items():
        for img_path, lbl_path in pairs:
            shutil.copy(img_path, f"{output_dir}/{split_name}/images/{os.path.basename(img_path)}")
            shutil.copy(lbl_path, f"{output_dir}/{split_name}/labels/{os.path.basename(lbl_path)}")
    
    # Create YAML configuration
    yaml_data = {
        'path': os.path.abspath(output_dir),
        'train': 'train/images',
        'val': 'val/images', 
        'test': 'test/images',
        'nc': 4,
        'names': ['Pothole', 'Alligator Crack', 'Transverse Crack', 'Longitudinal Crack']
    }
    
    with open(f"{output_dir}/data.yaml", 'w') as f:
        yaml.dump(yaml_data, f, sort_keys=False)
    
    return output_dir

# Prepare dataset if not exists
if not os.path.exists('yolo_dataset/data.yaml'):
    prepare_yolo_dataset()

In [4]:
# Prepare binary classification dataset for YOLO training (Road Damage vs No Damage)
def prepare_binary_yolo_dataset(data_dir="data", output_dir="yolo_binary_dataset"):
    """
    Prepare YOLO dataset for binary classification:
    - Class 0: Road Damage (any type of damage)
    - All original damage classes (0,1,2,3) will be converted to class 0
    """
    # Create directory structure
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)
    
    # Collect all image-label pairs from country directories
    all_pairs = []
    country_dirs = [d for d in os.listdir(data_dir) if d.startswith('country_')]
    
    for country_dir in country_dirs:
        images_path = f"{data_dir}/{country_dir}/images"
        labels_path = f"{data_dir}/{country_dir}/labels"
        
        for img_file in glob(f"{images_path}/*.jpg") + glob(f"{images_path}/*.png"):
            base_name = os.path.splitext(os.path.basename(img_file))[0]
            label_file = f"{labels_path}/{base_name}.txt"
            if os.path.exists(label_file):
                all_pairs.append((img_file, label_file))
    
    # Split dataset (70% train, 15% val, 15% test)
    random.seed(42)
    random.shuffle(all_pairs)
    
    n = len(all_pairs)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)
    
    splits = {
        'train': all_pairs[:train_end],
        'val': all_pairs[train_end:val_end],
        'test': all_pairs[val_end:]
    }
    
    # Copy files and convert labels to binary classification
    for split_name, pairs in splits.items():
        for img_path, lbl_path in pairs:
            # Copy image
            shutil.copy(img_path, f"{output_dir}/{split_name}/images/{os.path.basename(img_path)}")
            
            # Convert labels to binary (all damage classes become class 0)
            binary_label_path = f"{output_dir}/{split_name}/labels/{os.path.basename(lbl_path)}"
            
            with open(lbl_path, 'r') as f:
                lines = f.readlines()
            
            with open(binary_label_path, 'w') as f:
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        # Convert any damage class (0,1,2,3) to class 0 (damage)
                        class_id = int(parts[0])
                        if class_id in [0, 1, 2, 3]:  # Any damage type
                            parts[0] = '0'  # Convert to binary class 0 (damage)
                        # Keep the bounding box coordinates unchanged
                        f.write(' '.join(parts) + '\n')
    
    # Create YAML configuration for binary classification
    yaml_data = {
        'path': os.path.abspath(output_dir),
        'train': 'train/images',
        'val': 'val/images', 
        'test': 'test/images',
        'nc': 1,  # Only 1 class for binary classification
        'names': ['Road Damage']  # Class 0: Any road damage
    }
    
    with open(f"{output_dir}/data.yaml", 'w') as f:
        yaml.dump(yaml_data, f, sort_keys=False)
    
    # Print dataset statistics
    print(f"Binary YOLO dataset created at: {output_dir}")
    for split in ['train', 'val', 'test']:
        img_count = len(glob(f"{output_dir}/{split}/images/*.jpg") + glob(f"{output_dir}/{split}/images/*.png"))
        label_count = len(glob(f"{output_dir}/{split}/labels/*.txt"))
        print(f"{split}: {img_count} images, {label_count} labels")
    
    return output_dir

In [5]:
import cv2
import numpy as np
from PIL import Image, ImageEnhance

In [6]:
# Augmentation functions
def adjust_brightness(image, factor=1.2):
    return cv2.convertScaleAbs(image, alpha=factor, beta=0)

def adjust_contrast(image, factor=1.3):
    return cv2.convertScaleAbs(image, alpha=factor, beta=0)

def add_gaussian_noise(image, mean=0, std=10):
    noise = np.random.normal(mean, std, image.shape).astype(np.uint8)
    return cv2.add(image, noise)

def gaussian_blur(image, kernel_size=3):
    return cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)

def adjust_gamma(image, gamma=1.2):
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(image, table)

def adjust_saturation(image, factor=1.3):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hsv[:, :, 1] = cv2.multiply(hsv[:, :, 1], factor)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

In [7]:
def prepare_augmented_yolo_dataset(source_dir="yolo_dataset", output_dir="yolo_augmented_dataset", augment_factor=3, augment_test=False):
    """
    Prepare augmented YOLO dataset suitable for road damage detection
    """
    
    # Create directory structure
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)
    
    # Define augmentation combinations suitable for road damage
    augmentation_configs = [
        {'name': 'bright', 'funcs': [('brightness', {'factor': 1.3})]},
        {'name': 'dark', 'funcs': [('brightness', {'factor': 0.7})]},
        {'name': 'high_contrast', 'funcs': [('contrast', {'factor': 1.4})]},
        {'name': 'low_contrast', 'funcs': [('contrast', {'factor': 0.6})]},
        {'name': 'gamma_high', 'funcs': [('gamma', {'gamma': 1.5})]},
        {'name': 'gamma_low', 'funcs': [('gamma', {'gamma': 0.7})]},
        {'name': 'noise', 'funcs': [('noise', {'std': 8})]},
        {'name': 'blur', 'funcs': [('blur', {'kernel_size': 3})]},
        {'name': 'saturated', 'funcs': [('saturation', {'factor': 1.4})]},
        {'name': 'desaturated', 'funcs': [('saturation', {'factor': 0.6})]},
        {'name': 'bright_contrast', 'funcs': [('brightness', {'factor': 1.2}), ('contrast', {'factor': 1.2})]},
        {'name': 'dark_noise', 'funcs': [('brightness', {'factor': 0.9}), ('noise', {'std': 3})]},
    ]
    
    # Process each split
    for split in ['train', 'val', 'test']:
        print(f"Processing {split} split...")
        
        source_images_dir = f"{source_dir}/{split}/images"
        source_labels_dir = f"{source_dir}/{split}/labels"
        
        if not os.path.exists(source_images_dir):
            print(f"Source directory {source_images_dir} not found. Skipping {split}.")
            continue
        
        # Copy original images and labels
        image_files = glob(f"{source_images_dir}/*.jpg") + glob(f"{source_images_dir}/*.png")
        
        for img_path in image_files:
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            label_path = f"{source_labels_dir}/{base_name}.txt"
            
            if os.path.exists(label_path):
                shutil.copy(img_path, f"{output_dir}/{split}/images/{os.path.basename(img_path)}")
                shutil.copy(label_path, f"{output_dir}/{split}/labels/{os.path.basename(label_path)}")
                
                if split == 'test' and not augment_test:
                    continue
                
                image = cv2.imread(img_path)
                if image is None:
                    continue
                
                selected_augmentations = random.sample(augmentation_configs, min(augment_factor, len(augmentation_configs)))
                
                for aug_config in selected_augmentations:
                    augmented_image = image.copy()
                    
                    for func_name, params in aug_config['funcs']:
                        if func_name == 'brightness':
                            augmented_image = adjust_brightness(augmented_image, **params)
                        elif func_name == 'contrast':
                            augmented_image = adjust_contrast(augmented_image, **params)
                        elif func_name == 'gamma':
                            augmented_image = adjust_gamma(augmented_image, **params)
                        elif func_name == 'noise':
                            augmented_image = add_gaussian_noise(augmented_image, **params)
                        elif func_name == 'blur':
                            augmented_image = gaussian_blur(augmented_image, **params)
                        elif func_name == 'saturation':
                            augmented_image = adjust_saturation(augmented_image, **params)
                    
                    aug_img_name = f"{base_name}_{aug_config['name']}.jpg"
                    aug_label_name = f"{base_name}_{aug_config['name']}.txt"
                    
                    cv2.imwrite(f"{output_dir}/{split}/images/{aug_img_name}", augmented_image)
                    shutil.copy(label_path, f"{output_dir}/{split}/labels/{aug_label_name}")
    
    # Copy and update YAML configuration
    source_yaml = f"{source_dir}/data.yaml"
    if os.path.exists(source_yaml):
        with open(source_yaml, 'r') as f:
            yaml_data = yaml.safe_load(f)
        yaml_data['path'] = os.path.abspath(output_dir)
        with open(f"{output_dir}/data.yaml", 'w') as f:
            yaml.dump(yaml_data, f, sort_keys=False)
    
    print(f"Augmented dataset created at: {output_dir}")
    for split in ['train', 'val', 'test']:
        img_count = len(glob(f"{output_dir}/{split}/images/*.jpg") + glob(f"{output_dir}/{split}/images/*.png"))
        print(f"{split}: {img_count} images")
    
    return output_dir

In [8]:
def prepare_binary_augmented_yolo_dataset(source_dir="yolo_binary_dataset", output_dir="yolo_binary_augmented_dataset", augment_factor=3, augment_test=False):
    """
    Prepare augmented YOLO dataset for binary classification (road damage detection)
    """
    
    # Create directory structure
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)
    
    # Define augmentation combinations suitable for road damage
    augmentation_configs = [
        {'name': 'bright', 'funcs': [('brightness', {'factor': 1.3})]},
        {'name': 'dark', 'funcs': [('brightness', {'factor': 0.7})]},
        {'name': 'high_contrast', 'funcs': [('contrast', {'factor': 1.4})]},
        {'name': 'low_contrast', 'funcs': [('contrast', {'factor': 0.6})]},
        {'name': 'gamma_high', 'funcs': [('gamma', {'gamma': 1.5})]},
        {'name': 'gamma_low', 'funcs': [('gamma', {'gamma': 0.7})]},
        {'name': 'noise', 'funcs': [('noise', {'std': 8})]},
        {'name': 'blur', 'funcs': [('blur', {'kernel_size': 3})]},
        {'name': 'saturated', 'funcs': [('saturation', {'factor': 1.4})]},
        {'name': 'desaturated', 'funcs': [('saturation', {'factor': 0.6})]},
        {'name': 'bright_contrast', 'funcs': [('brightness', {'factor': 1.2}), ('contrast', {'factor': 1.2})]},
        {'name': 'dark_noise', 'funcs': [('brightness', {'factor': 0.9}), ('noise', {'std': 3})]},
    ]
    
    # Process each split
    for split in ['train', 'val', 'test']:
        print(f"Processing {split} split...")
        
        source_images_dir = f"{source_dir}/{split}/images"
        source_labels_dir = f"{source_dir}/{split}/labels"
        
        if not os.path.exists(source_images_dir):
            print(f"Source directory {source_images_dir} not found. Skipping {split}.")
            continue
        
        # Copy original images and labels
        image_files = glob(f"{source_images_dir}/*.jpg") + glob(f"{source_images_dir}/*.png")
        
        for img_path in image_files:
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            label_path = f"{source_labels_dir}/{base_name}.txt"
            
            if os.path.exists(label_path):
                shutil.copy(img_path, f"{output_dir}/{split}/images/{os.path.basename(img_path)}")
                shutil.copy(label_path, f"{output_dir}/{split}/labels/{os.path.basename(label_path)}")
                
                if split == 'test' and not augment_test:
                    continue
                
                image = cv2.imread(img_path)
                if image is None:
                    continue
                
                selected_augmentations = random.sample(augmentation_configs, min(augment_factor, len(augmentation_configs)))
                
                for aug_config in selected_augmentations:
                    augmented_image = image.copy()
                    
                    for func_name, params in aug_config['funcs']:
                        if func_name == 'brightness':
                            augmented_image = adjust_brightness(augmented_image, **params)
                        elif func_name == 'contrast':
                            augmented_image = adjust_contrast(augmented_image, **params)
                        elif func_name == 'gamma':
                            augmented_image = adjust_gamma(augmented_image, **params)
                        elif func_name == 'noise':
                            augmented_image = add_gaussian_noise(augmented_image, **params)
                        elif func_name == 'blur':
                            augmented_image = gaussian_blur(augmented_image, **params)
                        elif func_name == 'saturation':
                            augmented_image = adjust_saturation(augmented_image, **params)
                    
                    aug_img_name = f"{base_name}_{aug_config['name']}.jpg"
                    aug_label_name = f"{base_name}_{aug_config['name']}.txt"
                    
                    cv2.imwrite(f"{output_dir}/{split}/images/{aug_img_name}", augmented_image)
                    shutil.copy(label_path, f"{output_dir}/{split}/labels/{aug_label_name}")
    
    # Copy and update YAML configuration for binary classification
    source_yaml = f"{source_dir}/data.yaml"
    if os.path.exists(source_yaml):
        with open(source_yaml, 'r') as f:
            yaml_data = yaml.safe_load(f)
        yaml_data['path'] = os.path.abspath(output_dir)
        yaml_data['nc'] = 1  # Binary classification
        yaml_data['names'] = ['Road Damage']  # Only one class
        with open(f"{output_dir}/data.yaml", 'w') as f:
            yaml.dump(yaml_data, f, sort_keys=False)
    
    print(f"Binary augmented dataset created at: {output_dir}")
    for split in ['train', 'val', 'test']:
        img_count = len(glob(f"{output_dir}/{split}/images/*.jpg") + glob(f"{output_dir}/{split}/images/*.png"))
        print(f"{split}: {img_count} images")
    
    return output_dir

In [9]:
# Prepare datasets
if not os.path.exists('yolo_dataset/data.yaml'):
    prepare_yolo_dataset()

if not os.path.exists('yolo_augmented_dataset/data.yaml'):
    prepare_augmented_yolo_dataset(augment_test=False)

In [10]:
# Prepare binary classification datasets
print("Preparing binary classification datasets...")

# Prepare basic binary dataset
if not os.path.exists('yolo_binary_dataset/data.yaml'):
    prepare_binary_yolo_dataset()

# Prepare augmented binary dataset
if not os.path.exists('yolo_binary_augmented_dataset/data.yaml'):
    prepare_binary_augmented_yolo_dataset(augment_test=False)

Preparing binary classification datasets...
Binary YOLO dataset created at: yolo_binary_dataset
train: 4227 images, 4227 labels
val: 906 images, 906 labels
test: 906 images, 906 labels
Processing train split...
Binary YOLO dataset created at: yolo_binary_dataset
train: 4227 images, 4227 labels
val: 906 images, 906 labels
test: 906 images, 906 labels
Processing train split...
Processing val split...
Processing val split...
Processing test split...
Binary augmented dataset created at: yolo_binary_augmented_dataset
train: 16908 images
val: 3624 images
test: 906 images
Processing test split...
Binary augmented dataset created at: yolo_binary_augmented_dataset
train: 16908 images
val: 3624 images
test: 906 images
